In [1]:
# swagger : http://dev.jinwoosi.co.kr:8083/swagger-ui/index.html
# rest api :  http://dev.jinwoosi.co.kr:8083/v1/{데이터명}
# ex )  http://dev.jinwoosi.co.kr:8083/v1/air-quality?start_time=2023-06-09%2010:00:00&end_time=2023-06-09%2010:00:00

In [2]:
# http://dev.jinwoosi.co.kr:8083/v1/air-quality?start_date_time=2023-08-29%2000%3A00%3A00&end_date_time=2023-08-30%2000%3A00%3A00&page_number=1&number_of_rows=10&lang=ko

- start_date_time: 조회 시작일/시각
- end_date_time: 조회 종료일/시각
- page_number: 페이지 번호
- number_of_rows: 페이지 별 조회 개수

In [1]:
import urllib.parse
import requests
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder
from src.utils.util import *

In [2]:
# URL 디코딩
encoded_time = "%2000%3A00%3A00"
decoded_time = urllib.parse.unquote(encoded_time)
print(decoded_time)

# URL 인코딩 테스트
time_str = "2012:00:30" # 인코딩할 시간 값
encoded_time = urllib.parse.quote(time_str) # URL 인코딩
print(encoded_time)

 00:00:00
2012%3A00%3A30


## 데이터로드
- 2023년 05월 12일자부터 데이터 존재

#### 1. 대기 오염 정보

데이터 로드 테스트

In [2]:
# yyyy-MM-dd HH:mm:ss
start_time = "2023-09-01" + "%2001%3A00%3A00"
end_time = start_time
output_filename = 'test_data.json'
target = "air-quality"
data = fetch_data_from_url(target, start_time,end_time,output_filename)

HTTP 요청 성공


데이터 로드 및 데이터프레임 생성

In [4]:
# 2023년 05월 12일부터 오늘까지의 날짜를 가져와서 원하는 형태로 변환함
start_date_str = "2023-05-12"
end_date_str = datetime.now().strftime("%Y-%m-%d")
date_range = get_dates(start_date_str, end_date_str)
target = "air-quality"
observed_lst = fetch_data_from_url_for_timedelta(target,date_range)

100%|██████████| 165/165 [02:44<00:00,  1.01it/s]


저장 및 결측치 확인

In [ ]:
# 저장
df = pd.DataFrame(observed_lst)
df.to_csv("dataset/air_quality_data.csv",index=False)

# 결측치 확인
null_count = df.isna().sum()
print(null_count)

date_time                  0
measure_position_id        0
pm10_value              1568
pm25_value              3228
o3_flag                33567
pm10_flag              35863
pm25_flag              34203
o3_value                7167
so2_value              10436
co_value               10569
no2_value              10502
pm10_expected_value     9055
pm25_expected_value    10223
integrated_value       13291
integrated_grade       13291
so2_grade              10436
co_grade               10569
o3_grade               10730
no2_grade              10823
pm10_day_grade          9055
pm25_day_grade         10293
pm10_hour_grade         8918
pm25_hour_grade        27145
so2_flag               34336
co_flag                34203
no2_flag               34270
dtype: int64


시각화 확인

In [9]:
df = pd.read_csv("dataset/air_quality_data.csv")

In [9]:
# 위치 1~12 테스트 (pm 10)
"""
plt.figure(figsize=[16,12])
for test_pos in range(1,13):
    test_df = df[df["measure_position_id"]==test_pos]
    pm10 = test_df["pm10_value"].values
    null_count = test_df["pm10_value"].isna().sum()
    print(f"{test_pos}번 Position의 PM10 결측치 수 : {null_count}")
    time = np.arange(len(test_df["date_time"].values))
    
    plt.subplot(4,3,test_pos)
    plt.subplots_adjust(wspace=0.2, hspace=0.5)
    plt.plot(time,pm10)
    plt.title(f"position {test_pos}")
    plt.xlabel("time")
    plt.ylabel('pm10 value')
"""
# 위치 1~12 테스트 (pm 25)
"""
plt.figure(figsize=[24,16])
for test_pos in range(1,13):
    test_df = df[df["measure_position_id"]==test_pos]
    pm25 = test_df["pm25_value"].values
    null_count = test_df["pm25_value"].isna().sum()
    print(f"{test_pos}번 Position의 PM25 결측치 수 : {null_count}")
    time = np.arange(len(test_df["date_time"].values))
    
    plt.subplot(4,3,test_pos)
    plt.subplots_adjust(wspace=0.2, hspace=0.5)
    plt.plot(time,pm25)
    plt.title(f"position {test_pos}")
    plt.xlabel("time")
    plt.ylabel('pm25 value')
"""

'\nplt.figure(figsize=[24,16])\nfor test_pos in range(1,13):\n    test_df = df[df["measure_position_id"]==test_pos]\n    pm25 = test_df["pm25_value"].values\n    null_count = test_df["pm25_value"].isna().sum()\n    print(f"{test_pos}번 Position의 PM25 결측치 수 : {null_count}")\n    time = np.arange(len(test_df["date_time"].values))\n    \n    plt.subplot(4,3,test_pos)\n    plt.subplots_adjust(wspace=0.2, hspace=0.5)\n    plt.plot(time,pm25)\n    plt.title(f"position {test_pos}")\n    plt.xlabel("time")\n    plt.ylabel(\'pm25 value\')\n'

#### 2. 측정 위치 정보

데이터 로드 테스트

In [7]:
# yyyy-MM-dd HH:mm:ss
start_time = "2023-09-01" + "%2001%3A00%3A00"
end_time = start_time
output_filename = 'test_data.json'
target = "measure-position"
data = fetch_data_from_url(target, start_time,end_time,output_filename)

HTTP 요청 성공


데이터 로드 및 데이터프레임 생성

In [8]:
# 2023년 05월 12일부터 오늘까지의 날짜를 가져와서 원하는 형태로 변환함
start_date_str = "2023-05-12"
end_date_str = datetime.now().strftime("%Y-%m-%d")
date_range = get_dates(start_date_str, end_date_str)
target = "measure-position"
observed_lst = fetch_data_from_url_for_timedelta(target,date_range)

100%|██████████| 165/165 [02:48<00:00,  1.02s/it]


In [13]:
# 저장
df_position = pd.DataFrame(observed_lst)        
df_position.to_csv("dataset/position_data.csv",index=False)    

In [10]:
# 위치데이터 확인
df_position = pd.read_csv("dataset/position_data.csv")
df_1_12 = df_position[(df_position['id']>=1) & (df_position['id']<=12)].iloc[:12,:]
df_other = df_position[(df_position['id']>12)]
# plt.scatter(df_1_12['longitude'], df_1_12['latitude'], c='b', marker='o')
# plt.scatter(df_other['longitude'], df_other['latitude'], c='r', marker='o')
# plt.xlabel('Longitude')
# plt.ylabel('Latitude')
# plt.title('Point Plot')

#### 3. 환경 (풍속, 풍향 등) 정보확인
- 환경(풍속,풍향 등) 데이터는 8월 24일 06:00 부터 있음

In [ ]:
# yyyy-MM-dd HH:mm:ss
start_time = "2023-09-01" + "%2001%3A00%3A00"
end_time = start_time
output_filename = 'test_data.json'
target = "weather-vilage"
data = fetch_data_from_url(target, start_time,end_time,output_filename)

In [ ]:
# 2023년 05월 12일부터 오늘까지의 날짜를 가져와서 원하는 형태로 변환함
start_date_str = "2023-05-12"
end_date_str = datetime.now().strftime("%Y-%m-%d")
date_range = get_dates(start_date_str, end_date_str)
target = "weather-vilage"
observed_lst = fetch_data_from_url_for_timedelta(target,date_range)

In [17]:
# 저장
df_weather = pd.DataFrame(observed_lst)        
df_weather.to_csv("dataset/weather.csv",index=False)    

#### 4. 결합

In [29]:
# 로드
df_airquality = pd.read_csv("dataset/air_quality_data.csv") 
df_position = pd.read_csv("dataset/position_data.csv")
df_weather = pd.read_csv("dataset/weather.csv")

# label encoding
le = LabelEncoder()
df_weather['precipitation_one_hour'] = le.fit_transform(df_weather['precipitation_one_hour'])

# 각 데이터 지정
airquality = df_airquality[["date_time","measure_position_id","pm10_value","pm25_value"]]
position = df_position[(df_position['id']>=1) & (df_position['id']<=12)].iloc[:12,:]
# weather = df_weather[["date_time","measure_position_id","precipitation_one_hour","humidity","temperature","wind_speed",]]
weather = df_weather.drop(["precipitation_percent","snow_cover_one_hour",
                           "highest_temperature","lowest_temperature"],axis=1)


# 후처리
position.rename(columns={"id":"measure_position_id"},inplace=True) # 컬럼 이름변경
weather = weather.groupby('date_time').mean() # date_time 컬럼 기준으로 나머지 컬럼에 있는 값들 평균내기
weather.drop("measure_position_id",axis=1,inplace=True)

In [30]:
merged_df = pd.merge(airquality, position, on='measure_position_id', how='left')
merged_df = pd.merge(merged_df, weather, on='date_time', how='left')
merged_df['measure_position_id'] = merged_df['measure_position_id'].astype("float64")
merged_df.drop("grid_x",axis=1,inplace=True)
merged_df.drop("grid_y",axis=1,inplace=True)

In [11]:
merged_df.to_csv("dataset/merged_df.csv",index=False)

NameError: name 'merged_df' is not defined